In [ ]:
import pandas as pd
import numpy as np

file_path = "../../data/sms-call-internet-mi-2013-11-01.csv"

df = pd.read_csv(file_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nDtypes:")
print(df.dtypes)

Shape: (1891928, 8)

Columns:
['datetime', 'CellID', 'countrycode', 'smsin', 'smsout', 'callin', 'callout', 'internet']

Dtypes:
datetime        object
CellID           int64
countrycode      int64
smsin          float64
smsout         float64
callin         float64
callout        float64
internet       float64
dtype: object


In [3]:
print(df.head(20))
print("\n--- INFO ---")
df.info()

print("\n--- NULL COUNTS ---")
print(df.isnull().sum())

print("\n--- DUPLICATES ---")
print(df.duplicated().sum())

print("\n--- NEGATIVE VALUES ---")
activity_columns = [
    "smsin",
    "smsout",
    "callin",
    "callout",
    "internet"
]

for col in activity_columns:
    print(col, (df[col] < 0).sum())

               datetime  CellID  countrycode   smsin  smsout  callin  callout  \
0   2013-11-01 00:00:00       1            0  0.3521     NaN     NaN   0.0273   
1   2013-11-01 00:00:00       1           33     NaN     NaN     NaN      NaN   
2   2013-11-01 00:00:00       1           39  1.7322  1.1047  0.5919   0.4020   
3   2013-11-01 00:00:00       2            0  0.3581     NaN     NaN   0.0273   
4   2013-11-01 00:00:00       2           33     NaN     NaN     NaN      NaN   
5   2013-11-01 00:00:00       2           39  1.7334  1.0880  0.6020   0.4109   
6   2013-11-01 00:00:00       3            0  0.3644     NaN     NaN   0.0273   
7   2013-11-01 00:00:00       3           33     NaN     NaN     NaN      NaN   
8   2013-11-01 00:00:00       3           39  1.7348  1.0701  0.6128   0.4203   
9   2013-11-01 00:00:00       4            0  0.3349     NaN     NaN   0.0273   
10  2013-11-01 00:00:00       4           33     NaN     NaN     NaN      NaN   
11  2013-11-01 00:00:00     

In [4]:
column_mapping = {
    "datetime": "timestamp",
    "CellID": "grid_id",
    "countrycode": "country_code",
    "smsin": "sms_in",
    "smsout": "sms_out",
    "callin": "call_in",
    "callout": "call_out",
    "internet": "internet_activity"
}

canonical_df = df.rename(columns=column_mapping).copy()

print(canonical_df.columns.tolist())

['timestamp', 'grid_id', 'country_code', 'sms_in', 'sms_out', 'call_in', 'call_out', 'internet_activity']


In [5]:
canonical_df["timestamp"] = pd.to_datetime(
    canonical_df["timestamp"],
    errors="coerce"
)

canonical_df["grid_id"] = pd.to_numeric(
    canonical_df["grid_id"],
    errors="coerce"
).astype("Int64")

canonical_df["country_code"] = pd.to_numeric(
    canonical_df["country_code"],
    errors="coerce"
).astype("Int64")

for col in [
    "sms_in",
    "sms_out",
    "call_in",
    "call_out",
    "internet_activity"
]:
    canonical_df[col] = pd.to_numeric(
        canonical_df[col],
        errors="coerce"
    )
print(canonical_df.dtypes)

timestamp            datetime64[ns]
grid_id                       Int64
country_code                  Int64
sms_in                      float64
sms_out                     float64
call_in                     float64
call_out                    float64
internet_activity           float64
dtype: object


In [6]:
print("Invalid timestamps:",
      canonical_df["timestamp"].isna().sum())

print("Distinct timestamps:",
      canonical_df["timestamp"].nunique())

print("Minimum timestamp:",
      canonical_df["timestamp"].min())

print("Maximum timestamp:",
      canonical_df["timestamp"].max())

Invalid timestamps: 0
Distinct timestamps: 24
Minimum timestamp: 2013-11-01 00:00:00
Maximum timestamp: 2013-11-01 23:00:00


In [7]:
timestamps = (
    canonical_df["timestamp"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

print(timestamps)


0    2013-11-01 00:00:00
1    2013-11-01 01:00:00
2    2013-11-01 02:00:00
3    2013-11-01 03:00:00
4    2013-11-01 04:00:00
5    2013-11-01 05:00:00
6    2013-11-01 06:00:00
7    2013-11-01 07:00:00
8    2013-11-01 08:00:00
9    2013-11-01 09:00:00
10   2013-11-01 10:00:00
11   2013-11-01 11:00:00
12   2013-11-01 12:00:00
13   2013-11-01 13:00:00
14   2013-11-01 14:00:00
15   2013-11-01 15:00:00
16   2013-11-01 16:00:00
17   2013-11-01 17:00:00
18   2013-11-01 18:00:00
19   2013-11-01 19:00:00
20   2013-11-01 20:00:00
21   2013-11-01 21:00:00
22   2013-11-01 22:00:00
23   2013-11-01 23:00:00
Name: timestamp, dtype: datetime64[ns]


In [8]:
intervals = timestamps.diff().dropna()

print("\nTime intervals:")
print(intervals.value_counts())


Time intervals:
timestamp
0 days 01:00:00    23
Name: count, dtype: int64


In [9]:
expected_interval = pd.Timedelta(hours=1)

hourly_cadence = (
    len(timestamps) == 24
    and (intervals == expected_interval).all()
)

print("Exactly 24 timestamps:", len(timestamps) == 24)
print("All intervals are 1 hour:", (intervals == expected_interval).all())
print("Hourly cadence valid:", hourly_cadence)

Exactly 24 timestamps: True
All intervals are 1 hour: True
Hourly cadence valid: True


In [10]:
canonical_df["date"] = canonical_df["timestamp"].dt.date
canonical_df["hour"] = canonical_df["timestamp"].dt.hour
canonical_df["day_of_week"] = (
    canonical_df["timestamp"].dt.day_name()
)

In [11]:
null_counts = canonical_df.isnull().sum()

print(null_counts)

timestamp                  0
grid_id                    0
country_code               0
sms_in               1086153
sms_out              1422446
call_in              1407781
call_out             1037413
internet_activity    1087074
date                       0
hour                       0
day_of_week                0
dtype: int64


In [12]:
null_profile = pd.DataFrame({
    "null_count": canonical_df.isnull().sum(),
    "null_percentage": (
        canonical_df.isnull().mean() * 100
    ).round(2)
})

print(null_profile)

                   null_count  null_percentage
timestamp                   0             0.00
grid_id                     0             0.00
country_code                0             0.00
sms_in                1086153            57.41
sms_out               1422446            75.18
call_in               1407781            74.41
call_out              1037413            54.83
internet_activity     1087074            57.46
date                        0             0.00
hour                        0             0.00
day_of_week                 0             0.00


In [13]:
print(
    "Missing timestamp:",
    canonical_df["timestamp"].isna().sum()
)

print(
    "Missing grid_id:",
    canonical_df["grid_id"].isna().sum()
)

invalid_grid_ids = canonical_df[
    ~canonical_df["grid_id"].between(1, 10000)
]

print("Invalid grid IDs:", len(invalid_grid_ids))

exact_duplicates = canonical_df.duplicated().sum()

print("Exact duplicate rows:", exact_duplicates)

Missing timestamp: 0
Missing grid_id: 0
Invalid grid IDs: 0


Exact duplicate rows: 0


In [14]:
raw_key = [
    "timestamp",
    "grid_id",
    "country_code"
]
activity_columns = [
    "sms_in",
    "sms_out",
    "call_in",
    "call_out",
    "internet_activity"
]
raw_key_duplicates = (
    canonical_df.duplicated(subset=raw_key)
    .sum()
)

print(
    "Duplicate raw-grain rows:",
    raw_key_duplicates
)

negative_profile = {}

for col in activity_columns:
    negative_profile[col] = (
        canonical_df[col] < 0
    ).sum()

print(negative_profile)

country_counts = (
    canonical_df["country_code"]
    .value_counts(dropna=False)
    .sort_index()
)

print(country_counts)

Duplicate raw-grain rows: 0
{'sms_in': np.int64(0), 'sms_out': np.int64(0), 'call_in': np.int64(0), 'call_out': np.int64(0), 'internet_activity': np.int64(0)}
country_code
0        231307
1         42777
7         27509
20        78182
27         1260
          ...  
18768        13
68572        28
88235         9
88239      4259
97259        51
Name: count, Length: 246, dtype: Int64


In [15]:
canonical_df["total_sms"] = (
    canonical_df["sms_in"].fillna(0)
    + canonical_df["sms_out"].fillna(0)
)

canonical_df["total_calls"] = (
    canonical_df["call_in"].fillna(0)
    + canonical_df["call_out"].fillna(0)
)

canonical_df["total_activity"] = (
    canonical_df["total_sms"]
    + canonical_df["total_calls"]
    + canonical_df["internet_activity"].fillna(0)
)

In [16]:
hourly_activity = (
    canonical_df
    .groupby("timestamp", as_index=False)
    ["total_activity"]
    .sum()
    .sort_values("total_activity", ascending=False)
)

display(hourly_activity.head(10))

busiest_hour = hourly_activity.iloc[0]

print("Busiest timestamp:", busiest_hour["timestamp"])
print("Total activity:", busiest_hour["total_activity"])

,timestamp,total_activity
11,2013-11-01 11:00:00,5.185704e+06
18,2013-11-01 18:00:00,5.135660e+06
17,2013-11-01 17:00:00,5.108601e+06
12,2013-11-01 12:00:00,5.038444e+06
10,2013-11-01 10:00:00,4.926999e+06
16,2013-11-01 16:00:00,4.916011e+06
19,2013-11-01 19:00:00,4.912902e+06
15,2013-11-01 15:00:00,4.808330e+06
14,2013-11-01 14:00:00,4.774330e+06
20,2013-11-01 20:00:00,4.700756e+06


Busiest timestamp: 2013-11-01 11:00:00
Total activity: 5185703.659


In [17]:
grid_activity = (
    canonical_df
    .groupby("grid_id", as_index=False)
    ["total_activity"]
    .sum()
    .sort_values("total_activity", ascending=False)
)

display(grid_activity.head(10))

busiest_grid = grid_activity.iloc[0]

print("Busiest grid:", busiest_grid["grid_id"])
print("Total activity:", busiest_grid["total_activity"])

,grid_id,total_activity
5160,5161,274800.2956
5058,5059,197968.3438
5757,5758,143489.2707
5060,5061,137435.4867
5954,5955,135844.9556
5261,5262,131322.8759
6063,6064,126818.9053
5161,5162,124664.3217
4856,4857,121991.8232
4855,4856,114005.3370


Busiest grid: 5161.0
Total activity: 274800.2956


In [18]:
raw_counts = (
    canonical_df
    .groupby(["timestamp", "grid_id"])
    .size()
    .reset_index(name="country_code_rows")
)

print(raw_counts.head())

   timestamp  grid_id  country_code_rows
0 2013-11-01        1                  3
1 2013-11-01        2                  3
2 2013-11-01        3                  3
3 2013-11-01        4                  3
4 2013-11-01        5                  3


In [19]:
import json

geojson_path = "../data/reference/milano-grid.geojson"

with open(geojson_path, "r", encoding="utf-8") as f:
    geo = json.load(f)

features = geo["features"]

print("Feature count:", len(features))

Feature count: 10000


In [20]:
cell_ids = [
    feature["properties"]["cellId"]
    for feature in features
]

print("Minimum cellId:", min(cell_ids))
print("Maximum cellId:", max(cell_ids))
print("Unique cellIds:", len(set(cell_ids)))

Minimum cellId: 1
Maximum cellId: 10000
Unique cellIds: 10000
